# Speaker Feedback System - End-to-End Pipeline

This notebook runs speech, slide, and visual analyses to produce:
- per-slide recommendations (text + tables/figures captured from OCR)
- overall presentation feedback
- JSON and Markdown outputs for report/PDF generation


## 1. Setup and Imports
Load project modules and dependencies once so every later cell can reuse them.


In [ ]:
import os
import json
import gc
import logging
from pathlib import Path
from collections import defaultdict
from typing import Any, Dict, List, Optional

import numpy as np
import cv2
import torch
import imageio_ffmpeg

from agents.tools.tool_registry import speech_analysis_tool, slide_extraction_tool
from agents.tools.face_cache_tools import build_face_cache_tool
from agents.tools.clothing_tool import clothing_analysis_tool
from agents.tools.emotion_tool import emotion_analysis_tool
from agents.tools.gaze_tool import gaze_analysis_tool
from agents.tools.gesture_tool import gestures_analysis_tool
from agents.tools.recommendation_tool import build_recommendations

from slide_analysis.slide_content_parser import parse_slide_text
from video_analysis.gaze_estimator import MediaPipeGazeDirection
from video_analysis.gesture_analysis import Gestures
from video_analysis.clothing_model import ClothesCLIP

from emotiefflib.facial_analysis import EmotiEffLibRecognizer, get_model_list

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="torch")

print("Imports OK")


## 2. Model Cache and Paths
Set project-local cache paths and define input/output locations.


In [ ]:
MODEL_CACHE = Path("model_cache")
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["XDG_CACHE_HOME"] = str(MODEL_CACHE)

ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
os.environ["PATH"] = str(Path(ffmpeg_exe).parent) + os.pathsep + os.environ.get("PATH", "")

VIDEO_PATH = str(Path("video") / "VK_New_Video.mp4")
WEIGHTS_DIR = MODEL_CACHE / "weights"
DETECTRON_MODEL = str(WEIGHTS_DIR / "model_best.pth")
DETECTRON_CONFIG = str(WEIGHTS_DIR / "my_custom_config.yaml")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model cache:", MODEL_CACHE.resolve())
print("ffmpeg:", ffmpeg_exe)
print("Video:", VIDEO_PATH)
print("Detectron weights:", DETECTRON_MODEL)
print("Detectron config:", DETECTRON_CONFIG)
print("GPU available:", torch.cuda.is_available())


## 3. Helper Utilities
Lightweight helpers for cleanup and safe casting.


In [ ]:
def force_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Cleanup complete")

def safe_int(val, default=0):
    try:
        return int(val)
    except Exception:
        return default

def safe_float(val, default=0.0):
    try:
        return float(val)
    except Exception:
        return default


## 4. Speech Analysis
Transcribe the audio and compute filler words, intelligibility, noise, and pacing.


In [ ]:
print("Running speech analysis...")
speech_out = speech_analysis_tool(
    video_path=VIDEO_PATH,
    language="en",
    intelligibility_segment_len=30,
)

print("Speech keys:", list(speech_out.keys()))
print("Speech segments:", len(speech_out.get("segments", [])))

force_cleanup()


## 5. Slide Detection (SSIM + OCR)
Detect slide transitions visually, then refine with OCR to merge duplicate segments.


In [ ]:
print("Running slide extraction (SSIM + OCR refine)...")
slides_out = slide_extraction_tool(
    video_path=VIDEO_PATH,
    model_path=DETECTRON_MODEL,
    config_path=DETECTRON_CONFIG,
    ssim_thresh=0.70,
    min_segment_sec=10.0,
    similarity_threshold=0.78,
    min_word_count_for_slide=15,
)

final_slides = slides_out["segments"]
print("Raw segments:", slides_out["raw_count"])
print("Final slides:", slides_out["final_count"])

if final_slides:
    print("Slide 1 OCR preview:")
    print(final_slides[0].get("ocr_text", "")[:400])

force_cleanup()


## 6. Align Audio with Slides
Attach speech segments to each slide window for per-slide recommendations.


In [ ]:
print("Aligning speech segments to slides...")
speech_segments = speech_out.get("segments", [])

final_timeline = []
for slide in final_slides:
    s_start = safe_float(slide["start_time"])
    s_end = safe_float(slide["end_time"])
    overlap_text = []
    for seg in speech_segments:
        a_start = safe_float(seg.get("start", 0.0))
        a_end = safe_float(seg.get("end", 0.0))
        latest_start = max(s_start, a_start)
        earliest_end = min(s_end, a_end)
        if earliest_end > latest_start:
            overlap_text.append(seg.get("text", ""))

    final_timeline.append({
        "slide_id": slide["slide_id"],
        "start_time": s_start,
        "end_time": s_end,
        "duration": safe_float(slide.get("duration", s_end - s_start)),
        "visual_text": slide.get("ocr_text", ""),
        "visual_word_count": safe_int(slide.get("ocr_word_count", 0)),
        "spoken_text": " ".join(overlap_text).strip(),
    })

idx_to_slide = {
    int(seg["slide_id"]): {
        "slide_content": seg.get("visual_text", ""),
        "audio_content": seg.get("spoken_text", ""),
    }
    for seg in final_timeline
}

print("Timeline created:", len(final_timeline))


## 7. Parse OCR Content (Tables and Figures)
Extract tables/figures from OCR text to enrich slide recommendations.


In [ ]:
for seg in final_timeline:
    seg["ocr_parsed"] = parse_slide_text(seg.get("visual_text", ""))

table_count = sum(1 for s in final_timeline if s.get("ocr_parsed", {}).get("tables"))
figure_count = sum(1 for s in final_timeline if s.get("ocr_parsed", {}).get("figures"))
print(f"Detected tables in {table_count} slides and figures in {figure_count} slides.")

## 8. Persist Baseline JSON
Store speech, slide, and OCR-aligned timeline for downstream tasks.


In [ ]:
payload = {
    "meta": {
        "video_path": VIDEO_PATH,
        "total_slides": len(final_timeline),
    },
    "speech_stats": {
        "transcription": speech_out.get("transcription", ""),
        "filler_words": speech_out.get("filler_words", {}),
        "filler_phrases": speech_out.get("filler_phrases", {}),
        "speech_rate": speech_out.get("speech_rate", {}),
        "background_noise": speech_out.get("background_noise", []),
        "intelligibility": speech_out.get("intelligibility", []),
    },
    "slides_debug": {
        "raw_ssim_count": slides_out.get("raw_count", 0),
        "final_slide_count": slides_out.get("final_count", 0),
        "ssim_transitions": slides_out.get("ssim", {}).get("transitions", []),
        "ssim_params": slides_out.get("ssim", {}).get("params", {}),
    },
    "timeline": final_timeline,
}

output_file = OUTPUT_DIR / "presentation_analysis.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

print("Saved:", output_file.resolve())


## 9. Face Cache (Shared for Visual Analyses)
Sample frames per slide and cache face crops for downstream models.


In [ ]:
results = {
    "video_info": {"fps": None},
    "segments": final_timeline,
}

cache_out = build_face_cache_tool(
    video_path=VIDEO_PATH,
    segments=results["segments"],
    fps=results.get("video_info", {}).get("fps"),
    per_slide_frames=12,
    batch_size=24,
)

results["video_info"]["fps"] = cache_out["fps"]
results["slide_frame_mapping"] = cache_out["slide_frame_mapping"]
results["face_crops_cache"] = cache_out["face_crops_cache"]
results["face_cache_stats"] = cache_out["stats"]

print("Face cache stats:", results["face_cache_stats"])


## 10. Clothing Analysis (Overall)
CLIP-based attire classification; uses aggregated frames across slides.


In [ ]:
clothing_classifier = ClothesCLIP()
print(
    "Using CLIP model for clothing analysis."
    if (clothing_classifier.model and clothing_classifier.processor)
    else "Using fallback clothing analysis."
)

clothing_out = clothing_analysis_tool(
    video_path=VIDEO_PATH,
    slide_frame_mapping=results["slide_frame_mapping"],
    face_crops_cache=results["face_crops_cache"],
    clothing_classifier=clothing_classifier,
    frames_per_slide_max=4,
    min_face_conf=0.55,
)

results["clothing_analysis"] = {
    "is_appropriate": clothing_out["is_appropriate"],
    "detected_attributes": clothing_out["detected_attributes"],
    "recommendation": clothing_out["recommendation"],
    "coverage": clothing_out["coverage"],
}

print("Clothing analysis:", results["clothing_analysis"])

del clothing_classifier
force_cleanup()


## 11. Emotion Analysis
Per-slide emotion distribution and overall dominant emotion.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
fer = EmotiEffLibRecognizer(engine="onnx", model_name=get_model_list()[0], device=device)

emotion_out = emotion_analysis_tool(
    video_path=VIDEO_PATH,
    slide_frame_mapping=results["slide_frame_mapping"],
    face_crops_cache=results["face_crops_cache"],
    fer=fer,
    idx_to_slide=idx_to_slide,
    frames_per_slide_max=6,
    min_face_conf=0.55,
)

results["emotion_analysis"] = emotion_out
print("Overall emotion stats:", emotion_out.get("overall_stats", {}))

for seg in results["segments"]:
    sid = str(seg["slide_id"])
    slide_summary = emotion_out.get("slide_summaries", {}).get(sid, {})
    seg["dominant_emotion"] = slide_summary.get("dominant_emotion")
    seg["emotion_confidence"] = slide_summary.get("avg_confidence", 0.0)

del fer
force_cleanup()


## 12. Gaze Analysis (Overall)
Track head/gaze direction and summarize eye contact quality.


In [ ]:
gaze_estimator = MediaPipeGazeDirection()
gaze_out = gaze_analysis_tool(
    VIDEO_PATH,
    results["slide_frame_mapping"],
    results["face_crops_cache"],
    gaze_estimator,
    idx_to_slide,
)

results["gaze_analysis"] = gaze_out
print("Gaze summary:", gaze_out.get("overall_summary", {}))

gaze_estimator.close()


## 13. Gesture Analysis (Overall)
Pose-based body language stats and recommendations.


In [ ]:
import gdown

weights_path = Path("yolov8n-pose.pt")
if not weights_path.exists():
    file_id = "1qkEOE92d1we8Mp56I-0NsqTD603hsGoP"
    gdown.download(f"https://drive.google.com/uc?id={file_id}", output=str(weights_path), quiet=False)

gesture_detector = Gestures(str(weights_path))

gesture_out = gestures_analysis_tool(
    video_path=VIDEO_PATH,
    slide_frame_mapping=results["slide_frame_mapping"],
    gesture_detector=gesture_detector,
    idx_to_slide=idx_to_slide,
    frames_per_slide_max=6,
)

results["gesture_analysis"] = gesture_out
print("Gesture overall:", gesture_out.get("overall", {}).get("recommendations", {}))

force_cleanup()


## 14. Recommendations (NeMo ReAct)
Recommendations are generated only via NeMo Agent Toolkit (ReAct).
Use the NAT config at `speaker_feedback_nemo/configs/recommendations.yml` to run the agent.
Ensure `NVIDIA_API_KEY` is set in your environment.


In [ ]:
# One-time install for NAT ReAct workflow (langchain 0.1.x).
# Newer langchain versions remove langchain.schema, which NAT 1.3.1 expects.
# Run this cell once, then restart the kernel if packages change.
!uv pip install -c constraints.txt     "langchain==0.1.20"     "langchain-core==0.1.53"     "langchain-community==0.0.38"     "langchain-text-splitters==0.0.2"     "langsmith==0.1.147"



In [ ]:
import json
import os
import sys
import subprocess
from pathlib import Path

# Temporary session-only key override (do not commit real keys).
os.environ["NVIDIA_API_KEY"] = "YOUR_KEY_HERE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

RECOMMENDATIONS_PATH = OUTPUT_DIR / "nemo_recommendations.json"

final_payload = dict(payload)
final_payload.update(results)

nemo_payload_path = OUTPUT_DIR / "analysis_payload_for_nemo.json"
with open(nemo_payload_path, "w", encoding="utf-8") as f:
    json.dump(final_payload, f, indent=2, ensure_ascii=False)
print("Saved NeMo payload:", nemo_payload_path.resolve())

# Ensure the local tools package is discoverable by NAT.
repo_root = Path.cwd().resolve()
env = os.environ.copy()
env["PYTHONPATH"] = str(repo_root) + os.pathsep + env.get("PYTHONPATH", "")

# Run NAT from the active interpreter. (nat has no __main__ entry point.)
cmd = [
    sys.executable,
    "-m",
    "nat.cli.main",
    "run",
    "--config_file",
    "speaker_feedback_nemo/configs/recommendations.yml",
    "--input",
    "Use payload at outputs/analysis_payload_for_nemo.json to generate recommendations.",
]

with open(RECOMMENDATIONS_PATH, "w", encoding="utf-8") as out_f:
    result = subprocess.run(cmd, stdout=out_f, stderr=subprocess.PIPE, text=True, env=env)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("NAT run failed; see stderr above.")

if not RECOMMENDATIONS_PATH.exists() or RECOMMENDATIONS_PATH.stat().st_size == 0:
    raise RuntimeError("Missing NeMo recommendations JSON. NAT run produced no output.")

with open(RECOMMENDATIONS_PATH, "r", encoding="utf-8") as f:
    recommendations = json.load(f)

final_payload["recommendations"] = recommendations
print("Overall recommendations:", recommendations.get("overall", []))



## 15. Export Markdown Report
Create a slide-by-slide report that can be converted to PDF.


In [ ]:
def render_markdown_report(report: Dict[str, Any]) -> str:
    lines = []
    lines.append("# Presentation Feedback Report")
    lines.append("")

    overall = report.get("recommendations", {}).get("overall", [])
    lines.append("## Overall Recommendations")
    if overall:
        for rec in overall:
            lines.append(f"- {rec}")
    else:
        lines.append("- No overall recommendations generated.")
    lines.append("")

    lines.append("## Slides")
    per_slide = report.get("recommendations", {}).get("per_slide", {})

    for seg in report.get("timeline", []):
        sid = str(seg.get("slide_id"))
        lines.append(f"### Slide {sid}")
        lines.append(f"- Time: {seg.get('start_time', 0):.1f}s - {seg.get('end_time', 0):.1f}s")

        ocr = seg.get("ocr_parsed", {})
        if ocr.get("clean_text"):
            lines.append("")
            lines.append("**Slide Text**")
            lines.append(ocr.get("clean_text", "")[:800])

        if ocr.get("tables"):
            lines.append("")
            lines.append("**Tables**")
            for tbl in ocr.get("tables", []):
                lines.append(tbl.get("markdown", ""))
                lines.append("")

        if ocr.get("figures"):
            lines.append("**Figures**")
            for fig in ocr.get("figures", []):
                lines.append(f"- {fig}")

        rec = per_slide.get(sid, {})
        strengths = rec.get("strengths", [])
        improvements = rec.get("improvements", [])

        if strengths:
            lines.append("")
            lines.append("**Strengths**")
            for s in strengths:
                lines.append(f"- {s}")

        if improvements:
            lines.append("")
            lines.append("**Improvements**")
            for s in improvements:
                lines.append(f"- {s}")

        lines.append("")

    return "\n".join(lines)

report_md = render_markdown_report(final_payload)
report_path = OUTPUT_DIR / "presentation_report.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_md)

json_path = OUTPUT_DIR / "presentation_report.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(final_payload, f, indent=2, ensure_ascii=False)

print("Saved Markdown:", report_path.resolve())
print("Saved JSON:", json_path.resolve())


## 16. PDF Export (Optional)
Convert `outputs/presentation_report.md` to PDF using your preferred tool (for example, pandoc or a Markdown-to-PDF pipeline).
